In [1]:
import requests
import json
import pandas as pd
import re
import os
import time

In [2]:
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
raw = '0 - raw'
path_raw = f'./{raw}'

In [4]:
bronze = '1 - bronze'
path_bronze = f'./{bronze}'

In [5]:
api_url = 'https://pokeapi.co/api/v2'

In [5]:
def create_folder(path):
    if not os.path.exists(path):
       os.makedirs(path)

In [6]:
def get_json(url, params={}):
    try:
        response = requests.get(url, params)
        return response.json()
        
    except Exception as error:
        print("An exception occurred:", error)

In [7]:
def save_json(path, filename, data):
    try:
        with open(f'{path}/{filename}.json', 'w') as f:
            json.dump(data, f)
            print(f'file {filename} saved')
            
    except Exception as error:
        print("An exception occurred:", error)

In [8]:
def load_jsonfile(path, name):
    with open(f'{path}/{name}', 'r') as f:
        return json.load(f)

In [10]:
def get_url_list(data_jsons):
    if len(data_jsons)==0:
        return []

    return [ x['url'] for x in data_jsons['results'] ]

In [11]:
def get_resource_name(url):
    try:
        name = re.search(r'/(\d+)', url).group(1)
    except Exception as error:
        print("An exception occurred:", error)
        name = 'default'
        
    return name

In [22]:
def download_and_save_data(table):

    # leer antiguo fichero de urls
    try:
        old_data = []
        old_count = 0
        
        old_data = load_jsonfile(path_raw, f'{table}.json')
        old_count = old_data['count']
        print(f'Encontrado archivo local: {old_count} recursos')
    except FileNotFoundError:
        print(f'No se ha encontrado un archivo anterior')

    # obtener número de recursos disponibles en la api
    data = get_json( 
        url = f'{api_url}/{table}'
    )
    count = data['count']
    print(f'Recursos disponibles en la API: {count} recursos')

    if old_count==count:
        return
    
    # obtener todas las urls
    data_jsons = get_json(
        url = f'{api_url}/{table}', 
        params = {'offset':0, 'limit':count}
    )

    # lista de urls anterior y nueva
    old_url_list = get_url_list(old_data)
    new_url_list = get_url_list(data_jsons)

    # obtener lista de urls de recursos a descargar
    url_list = list(set(new_url_list) - set(old_url_list))

    if len(url_list)==0:
        return

    print('')
    
    # crear carpeta si no existe
    #path = path_raw + '/new_data'
    #create_folder(path)

    # descargar los datos de cada url y guardarlos en una misma lista
    resource_data_list = []
    
    for url in url_list:
        # download data
        resource_data = get_json(url)
        filename = get_resource_name(url)
        print(f'- file {filename} extracted')
        
        resource_data_list.append(resource_data)
        
        # save file
        #save_json(path, filename, resource_data)

        # wait
        time.sleep(2)

    print('')

    # guardar archivo nuevo de urls
    save_json(path_raw, table, data_jsons)
    
    # guardar dataframe con los datos nuevos
    df = (
        pd.DataFrame(resource_data_list)
        .sort_values(by='id')
        .set_index('id', drop=False)
    )
    path_bronze_new_data = f'{path_bronze}/new_data/{table}.parquet'
    df.to_parquet(path_bronze_new_data)
    print(f'dataframe saved in {path_bronze_new_data}')

---
## DESCARGA DE DATOS

In [ ]:
# save all pokemon

In [17]:
download_and_save_data('pokemon')

Encontrado archivo anterior: 1351 recursos
Recursos disponibles en la API: 1351 recursos


In [18]:
# save all pokemon forms

In [19]:
download_and_save_data('pokemon-form')

Encontrado archivo anterior: 1579 recursos
Recursos disponibles en la API: 1579 recursos


In [20]:
# save all pokemon species

In [21]:
download_and_save_data('pokemon-species')

Encontrado archivo anterior: 1025 recursos
Recursos disponibles en la API: 1025 recursos


In [10]:
dfb = pd.read_parquet(f'{path_bronze}/backup/pokemon.parquet')

In [11]:
dfb.iloc[[0,-1]]

,abilities,base_experience,cries,forms,game_indices,height,held_items,id,is_default,location_area_encounters,moves,name,order,past_abilities,past_types,species,sprites,stats,types,weight
id,,,,,,,,,,,,,,,,,,,,
1,"[{'ability': {'name': 'overgrow', 'url': 'http...",64,{'latest': 'https://raw.githubusercontent.com/...,"[{'name': 'bulbasaur', 'url': 'https://pokeapi...","[{'game_index': 153, 'version': {'name': 'red'...",7,[],1,True,https://pokeapi.co/api/v2/pokemon/1/encounters,"[{'move': {'name': 'razor-wind', 'url': 'https...",bulbasaur,1,"[{'abilities': [{'ability': None, 'is_hidden':...",[],"{'name': 'bulbasaur', 'url': 'https://pokeapi....",{'back_default': 'https://raw.githubuserconten...,"[{'base_stat': 45, 'effort': 0, 'stat': {'name...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",69
10277,"[{'ability': {'name': 'teraform-zero', 'url': ...",90,{'latest': 'https://raw.githubusercontent.com/...,"[{'name': 'terapagos-stellar', 'url': 'https:/...",[],17,[],10277,False,https://pokeapi.co/api/v2/pokemon/10277/encoun...,"[{'move': {'name': 'headbutt', 'url': 'https:/...",terapagos-stellar,-1,[],[],"{'name': 'terapagos', 'url': 'https://pokeapi....","{'back_default': None, 'back_female': None, 'b...","[{'base_stat': 160, 'effort': 3, 'stat': {'nam...","[{'slot': 1, 'type': {'name': 'normal', 'url':...",770


In [6]:
df = pd.read_parquet(f'{path_bronze}/new_data/pokemon.parquet')

In [7]:
df.iloc[[0,-1]]

,abilities,base_experience,cries,forms,game_indices,height,held_items,id,is_default,location_area_encounters,moves,name,order,past_abilities,past_stats,past_types,species,sprites,stats,types,weight
id,,,,,,,,,,,,,,,,,,,,,
10278,"[{'ability': {'name': 'magic-bounce', 'url': '...",None,{'latest': 'https://raw.githubusercontent.com/...,"[{'name': 'clefable-mega', 'url': 'https://pok...",[],17,[],10278,False,https://pokeapi.co/api/v2/pokemon/10278/encoun...,"[{'move': {'name': 'fire-punch', 'url': 'https...",clefable-mega,-1,[],[],[],"{'name': 'clefable', 'url': 'https://pokeapi.c...",{'back_default': 'https://raw.githubuserconten...,"[{'base_stat': 95, 'effort': 0, 'stat': {'name...","[{'slot': 1, 'type': {'name': 'fairy', 'url': ...",423
10326,"[{'ability': {'name': 'trace', 'url': 'https:/...",None,{'latest': 'https://raw.githubusercontent.com/...,"[{'name': 'meowstic-female-mega', 'url': 'http...",[],8,[],10326,False,https://pokeapi.co/api/v2/pokemon/10326/encoun...,"[{'move': {'name': 'hyper-beam', 'url': 'https...",meowstic-female-mega,-1,[],[],[],"{'name': 'meowstic', 'url': 'https://pokeapi.c...",{'back_default': 'https://raw.githubuserconten...,"[{'base_stat': 74, 'effort': 0, 'stat': {'name...","[{'slot': 1, 'type': {'name': 'psychic', 'url'...",101
